# RLHF with PPO — Aligning an LLM to *Human* Preferences

## What is RLHF?

**RLHF (Reinforcement Learning from Human Feedback)** is the alignment recipe behind InstructGPT/ChatGPT. A **Reward Model** is trained to imitate **human** pairwise judgments (`chosen` vs `rejected`), then **PPO** optimizes the policy to maximize that reward — with a **KL penalty** anchoring it to the supervised (SFT) starting point so it stays fluent.

> 🔑 **Why this notebook is genuine RLHF:** its reward model is trained on **`Anthropic/hh-rlhf`**, whose `chosen`/`rejected` labels come from **real human crowdworkers** comparing assistant responses for helpfulness & harmlessness. Contrast this with the companion **`RLAIF_with_PPO.ipynb`**, which uses the *same machinery* but trains the reward model on **AI-generated** (GPT-4) preferences. The only difference between RLHF and RLAIF is **who produced the preference labels**.

## The Three Phases of RLHF

| Phase | What | This notebook |
|---|---|---|
| **1. SFT** | A base LM is supervised-fine-tuned to follow instructions ("learns *how* to talk") | We reuse **`Qwen2.5-0.5B-Instruct`**, which is already SFT'd |
| **2. Reward Modeling** | Train a reward model on **human** preference pairs ("learns *what humans value*") | `RewardTrainer` on **`Anthropic/hh-rlhf`** |
| **3. PPO** | RL-optimize the policy against the frozen reward model, KL-regularized | `PPOTrainer` on hh-rlhf prompts |

### The KL-Penalty Objective

$$R_{\text{total}}(x, y) = R_{\phi}(x, y) - \beta \log \left( \frac{\pi_{\theta}(y|x)}{\pi_{\text{ref}}(y|x)} \right)$$

where $R_{\phi}$ is the **human-preference** reward and the $\beta$-scaled KL term keeps the policy $\pi_\theta$ close to the frozen reference $\pi_{\text{ref}}$ (the SFT model), preventing reward hacking.

> **Reward Model vs. Critic — do not conflate them.** The **Reward Model** encodes *human* preferences and is **frozen** during PPO. The **Critic** (value head) is learned *online during PPO* to estimate returns for the advantage computation. Two different models, two different jobs.

---

## VRAM & Compute Impact (Colab T4, 16 GB)

PPO keeps up to **four models** resident (Actor, Critic, Reward, Reference). We fit them on a single **16 GB T4** via **4-bit QLoRA**, and remove the 4th (Reference) by disabling the policy's LoRA adapters on the fly (`ref_model=None`).

# Production-Grade Implementation (Colab T4, 16 GB)

**4-bit Quantization (QLoRA)** for every model + TRL's adapter-disabling reference trick.

> ⚙️ **Reference model via adapter disabling:** `ref_model=None` on a PEFT policy makes TRL compute the KL penalty by temporarily disabling the LoRA adapters — the frozen base weights *are* the reference. One fewer model in VRAM.

**Pipeline (5 executable stages, mapping onto the 3 RLHF phases above):**

| Stage | What | RLHF phase |
|---|---|---|
| 1 | `Qwen2.5-0.5B-Instruct` as the policy | Phase 1 (SFT, pre-done) |
| 2 | Train a **Reward Model** on `Anthropic/hh-rlhf` **human** pairs | Phase 2 |
| 3 | Assemble Actor + Critic + Reward + (implicit) Reference | Phase 3 setup |
| 4 | Prepare **prompt-only** PPO data (hh-rlhf user turns) | Phase 3 setup |
| 5 | **PPO** train, then evaluate | Phase 3 |


## Environment Setup

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
%pip install torchao==0.16.0 transformers trl peft accelerate bitsandbytes datasets

In [ ]:
import torch
import os
import gc
import re
from transformers import AutoTokenizer, DataCollatorWithPadding, BitsAndBytesConfig, set_seed, AutoModelForCausalLM, AutoModelForSequenceClassification, GenerationConfig
from trl import RewardConfig, RewardTrainer
from trl.experimental.ppo import PPOConfig, PPOTrainer
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset

set_seed(42)
# Prevents CUDA fragmentation OOMs on Colab T4
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## Setup — Quantization, LoRA & Tokenizer

We start from the **Instruct** checkpoint (Phase 1 / SFT is already done — Qwen fine-tuned it to follow instructions), define one 4-bit config shared by every model, and two LoRA configs: one for the **policy** (a Causal LM) and one for the **reward model** (a Sequence-Classification model, which additionally needs its `score` head trained).

In [ ]:
# Start from the *Instruct* checkpoint: RLHF Phase 1 (SFT) is already done here.
# It can already follow instructions and answer helpfully, giving PPO coherent
# behaviour to reinforce AND a coherent KL reference.
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. 4-bit Quantization (shared by every model to fit the T4's 16 GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 2a. LoRA for the POLICY (a CausalLM) — no score head here.
policy_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)

# 2b. LoRA for the REWARD MODEL (a SequenceClassification model).
#     modules_to_save=["score"] is ESSENTIAL: the scalar reward head starts random,
#     so it must be trainable (not frozen) or the reward signal stays meaningless.
reward_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    bias="none",
    task_type="SEQ_CLS",
    modules_to_save=["score"],
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Stage 2 — Reward Model Training (on **Human** Feedback)

This is what makes it RLHF: the reward model learns from **`Anthropic/hh-rlhf`**, real human `(chosen, rejected)` judgments, under the **Bradley–Terry** objective (maximize the score gap between the human-preferred and rejected response).

**Format note.** hh-rlhf stores each example as a raw transcript string:

```
\n\nHuman: <question>\n\nAssistant: <answer>\n\nHuman: ...\n\nAssistant: ...
```

We parse those into chat **messages** (`[{"role": "user"/"assistant", ...}]`) so TRL applies the **same Qwen chat template** the policy uses at PPO/inference time. Keeping the reward model and the policy in one consistent format is what makes the reward signal meaningful for the text PPO actually generates.

Watch the logged **`accuracy`** rise above `0.5` — proof the head learned a real human-preference signal, not noise.

In [ ]:
# --- Parse Anthropic HH-RLHF transcripts into chat-template messages ----------
# Each 'chosen'/'rejected' is a raw string:
#   "\n\nHuman: <q>\n\nAssistant: <a>\n\nHuman: ...\n\nAssistant: ..."
# Convert to [{"role": "user"/"assistant", "content": ...}, ...] so TRL applies the
# SAME Qwen chat template used by the policy — reward model and policy stay aligned.
def hh_to_messages(text):
    turns = re.split(r"\n\n(Human|Assistant): ", text)
    msgs = []
    for i in range(1, len(turns) - 1, 2):
        role = "user" if turns[i] == "Human" else "assistant"
        msgs.append({"role": role, "content": turns[i + 1].strip()})
    return msgs

def is_wellformed(msgs):
    # Qwen's chat template needs: non-empty, starts with 'user', strictly alternates,
    # ends with 'assistant'. Drop anything malformed so apply_chat_template can't fail.
    if len(msgs) < 2:
        return False
    for i, m in enumerate(msgs):
        expected = "user" if i % 2 == 0 else "assistant"
        if m["role"] != expected or not m["content"]:
            return False
    return msgs[-1]["role"] == "assistant"

# Over-select, then filter to well-formed pairs, then take a small T4-sized slice.
raw_rm = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=42).select(range(3000))

def format_hh(ex):
    return {"chosen": hh_to_messages(ex["chosen"]), "rejected": hh_to_messages(ex["rejected"])}

rm_dataset = raw_rm.map(format_hh, remove_columns=raw_rm.column_names)
rm_dataset = rm_dataset.filter(lambda ex: is_wellformed(ex["chosen"]) and is_wellformed(ex["rejected"]))
rm_dataset = rm_dataset.select(range(min(800, len(rm_dataset))))
print(f"Reward-model training pairs after filtering: {len(rm_dataset)}")

reward_config = RewardConfig(
    output_dir="./reward_model_adapter",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=1e-4,  # higher LR is fine — only LoRA + head are trained
    num_train_epochs=1,
    max_length=512,  # filters long multi-turn pairs; keeps VRAM in budget
    logging_steps=10,
    bf16=True, fp16=False,  # T4 runs bf16 (no GradScaler) -> avoids the fp16 "unscale
                            # not implemented for BFloat16" crash.
    center_rewards_coefficient=1e-2,  # keeps rewards ~zero-centered, curbs reward hacking
    report_to="none",
)

reward_trainer = RewardTrainer(
    model=base_model_id,  # loaded as AutoModelForSequenceClassification, num_labels=1
    args=reward_config,
    train_dataset=rm_dataset,
    processing_class=tokenizer,
    peft_config=reward_lora_config,  # trains LoRA + the "score" head
    quantization_config=bnb_config,  # QLoRA: 4-bit base
)
reward_trainer.train()

# 'accuracy' in the logs should climb well above 0.5 — a real human-preference signal.
reward_trainer.save_model("./reward_model_adapter")

# Free the trainer/optimizer state before PPO loads its 3 models.
del reward_trainer
gc.collect()
torch.cuda.empty_cache()

## Stage 3 — Assemble the PPO Models

Three models (the fourth — the **reference** — is the policy with adapters disabled, via `ref_model=None`):

- **Actor / Policy** — Instruct Causal LM + trainable LoRA.
- **Critic / Value model** — a fresh Sequence-Classification head. A *random* value head is **correct**; it is learned during PPO.
- **Reward model** — the human-preference model from Stage 2, loaded frozen.

In [ ]:
# A) ACTOR (Policy): Instruct CausalLM + QLoRA adapters.
#    ref_model=None => TRL reuses this model with adapters DISABLED as the frozen
#    KL reference (the coherent Instruct model).
actor_model = AutoModelForCausalLM.from_pretrained(
    base_model_id, quantization_config=bnb_config, device_map="auto"
)
actor_model = get_peft_model(actor_model, policy_lora_config)
actor_model.config.use_cache = False
actor_model.config.pad_token_id = tokenizer.pad_token_id
actor_model.generation_config = GenerationConfig(
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id
)

# B) CRITIC (Value Model): fresh SequenceClassification head.
#    A RANDOM value head is CORRECT here — the value function is learned during PPO.
value_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_id, num_labels=1, quantization_config=bnb_config, device_map="auto"
)
value_model.config.pad_token_id = tokenizer.pad_token_id  # seq-cls head needs this to
value_model.config.use_cache = False                      # find the last non-pad token

# C) REWARD MODEL: the human-preference model trained in Stage 2 (base + LoRA + score head).
#    Frozen during PPO. This is the real, human-aligned reward signal.
reward_base = AutoModelForSequenceClassification.from_pretrained(
    base_model_id, num_labels=1, quantization_config=bnb_config, device_map="auto"
)
reward_model = PeftModel.from_pretrained(reward_base, "./reward_model_adapter")
reward_model.eval()
reward_model.config.pad_token_id = tokenizer.pad_token_id
reward_model.config.use_cache = False

## Stage 4 — Dataset Preparation (PPO prompts)

PPO consumes **prompt-only** data. We reuse `Anthropic/hh-rlhf` but keep only the **first human turn** of each conversation as the prompt the policy must answer helpfully. Each is wrapped in the **Instruct chat template** (`add_generation_prompt=True`) so the policy emits a proper assistant turn — exactly the format the reward model was trained on.

In [ ]:
# Prompt-only PPO data: the first human turn of each hh-rlhf conversation.
raw_ppo = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=123).select(range(2000))

def build_prompt(example):
    msgs = hh_to_messages(example["chosen"])
    prompt_msgs = msgs[:1]  # single-turn: the first user question
    ok = bool(prompt_msgs) and prompt_msgs[0]["role"] == "user" and bool(prompt_msgs[0]["content"])
    if not ok:
        # placeholder; filtered out below
        return {"input_ids": [], "attention_mask": [], "keep": False}
    text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, truncation=True, max_length=384)
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "keep": True}

tokenized_dataset = raw_ppo.map(build_prompt, remove_columns=raw_ppo.column_names)
tokenized_dataset = tokenized_dataset.filter(lambda ex: ex["keep"])
tokenized_dataset = tokenized_dataset.select_columns(["input_ids", "attention_mask"])
tokenized_dataset = tokenized_dataset.select(range(min(300, len(tokenized_dataset))))
print(f"PPO prompts: {len(tokenized_dataset)}")

# Dynamic padding collator (saves VRAM vs. padding to max_length)
collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")

## Stage 5 — PPO Training

In [ ]:
# PPO Config
ppo_config = PPOConfig(
    # --- Standard TrainingArguments Base Parameters ---
    output_dir="./ppo_output",
    run_name="ppo-rlhf-t4",
    learning_rate=1e-5,      # low enough to be stable, high enough to move the policy
    logging_steps=1,

    # --- Hardened VRAM Constraints (Base Class) ---
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=True, fp16=False,   # bf16 => no fp16 GradScaler => no bf16-unscale crash

    # --- TRL Rollout & Generation Parameters ---
    local_rollout_forward_batch_size=2,
    response_length=64,
    temperature=0.7,
    stop_token="eos",
    missing_eos_penalty=1.0,  # nudges completions to actually terminate

    # --- TRL PPO Mathematical Step Parameters ---
    num_mini_batches=1,
    num_ppo_epochs=2,
    whiten_rewards=True,
    num_train_epochs=1,

    # --- Alignment & Trust Region Constraints ---
    kl_coef=0.1,  # lets the policy improve while the Instruct reference keeps it coherent
    kl_estimator="k3",
    cliprange=0.2,
    cliprange_value=0.2,
    vf_coef=0.1,
    gamma=1.0,
    lam=0.95,
)

# TRAINER INIT
ppo_trainer = PPOTrainer(
    args=ppo_config,
    model=actor_model,
    ref_model=None,  # None => KL reference = Instruct base (adapters disabled)
    reward_model=reward_model,
    value_model=value_model,
    processing_class=tokenizer,
    train_dataset=tokenized_dataset,
    data_collator=collator,
)

In [ ]:
ppo_trainer.train()

# Save the aligned adapter
ppo_trainer.save_model("./rlhf_with_ppo_adapter")

## Export — Download the Fine-Tuned Adapter (Optional)

In [ ]:
import shutil
import os

folder_to_zip = './rlhf_with_ppo_adapter'
output_filename = 'rlhf_with_ppo_adapter.zip'

shutil.make_archive(output_filename.replace('.zip', ''), 'zip', folder_to_zip)

if os.path.exists(output_filename):
    file_size = os.path.getsize(output_filename)
    print(f"File: {output_filename}")
    print(f"Size in MB: {file_size / (1024 * 1024):.2f} MB")
else:
    print(f"File {output_filename} not found. Run the zipping step above first.")

### Download the file to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Save the Adapter to Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

destination_folder = '/content/drive/MyDrive/colab_models'
if os.path.exists(output_filename):
    os.makedirs(destination_folder, exist_ok=True)
    destination_path = os.path.join(destination_folder, output_filename)
    print(f"Copying to Drive: {destination_path}...")
    shutil.copy(output_filename, destination_path)
    print("Done! The adapter is backed up to your Google Drive.")
else:
    print(f"Error: {output_filename} not found. Did training + zipping finish?")

# Model Usage — Evaluate the Aligned Policy

Reload the **Instruct** base + the PPO-tuned LoRA adapter, and generate with the **same chat-template formatting** used during training. (A format mismatch alone — feeding raw prompts to an Instruct model — produces incoherent output regardless of training quality.)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "./rlhf_with_ppo_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading base tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto"
)

print("Applying PPO-tuned LoRA adapters...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

In [ ]:
# INFERENCE UTILITY — same chat template the PPO loop used (a single user turn).
def generate_response(user_prompt, max_new_tokens=200, temperature=0.7):
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)

In [ ]:
# RUN TEST PROMPTS (helpfulness — the hh-rlhf objective)
test_prompts = [
    "What are three practical tips for staying focused while working from home?",
    "My close friend has been feeling really down lately. How can I support them?",
]

print("\n--- Generating RLHF-Aligned Responses ---")
for i, prompt in enumerate(test_prompts):
    print(f"\n[Test Prompt {i+1}]: {prompt}")
    print(f"[RLHF Aligned Response]: {generate_response(prompt).strip()}")
    print("-" * 50)